In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Quantencomputer für den Transpiler darstellen


<details>
<summary><b>Paketversionen</b></summary>

Der Code auf dieser Seite wurde mit den folgenden Anforderungen entwickelt.
Wir empfehlen, diese Versionen oder neuere zu verwenden.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Um einen abstrakten Circuit in einen ISA-Circuit umzuwandeln, der auf einem bestimmten QPU (Quantenprozessor) laufen kann, benötigt der Transpiler bestimmte Informationen über den QPU. Diese Informationen befinden sich an zwei Stellen: im `BackendV2`-Objekt (oder dem älteren `BackendV1`), an das du Jobs übermitteln möchtest, sowie im `Target`-Attribut des Backends.

- Das [`Target`](../api/qiskit/qiskit.transpiler.Target) enthält alle relevanten Einschränkungen eines Geräts, wie z. B. seine nativen Basis-Gates, die Qubit-Konnektivität sowie Puls- oder Timing-Informationen.
- Das [`Backend`](../api/qiskit/qiskit.providers.BackendV2) besitzt standardmäßig ein `Target`, enthält zusätzliche Informationen — wie die [`InstructionScheduleMap`](https://docs.quantum.ibm.com/api/qiskit/1.4/qiskit.pulse.InstructionScheduleMap) — und stellt die Schnittstelle zur Übermittlung von Quantum-Circuit-Jobs bereit.

Du kannst dem Transpiler auch explizit Informationen zur Verfügung stellen, zum Beispiel wenn du einen speziellen Anwendungsfall hast oder wenn du glaubst, dass diese Informationen dem Transpiler helfen, einen besser optimierten Circuit zu erzeugen.

Die Genauigkeit, mit der der Transpiler den am besten geeigneten Circuit für bestimmte Hardware erzeugt, hängt davon ab, wie viele Informationen das `Target` oder das `Backend` über seine Einschränkungen enthält.

> **Note:** Da viele der zugrunde liegenden Transpilationsalgorithmen stochastisch sind, gibt es keine Garantie, dass ein besserer Circuit gefunden wird.

Diese Seite zeigt mehrere Beispiele für die Übergabe von QPU-Informationen an den Transpiler. Diese Beispiele verwenden das Target des Mock-Backends [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke).

<span id="default-config"></span>
## Standardkonfiguration
Die einfachste Verwendung des Transpilers besteht darin, alle QPU-Informationen durch Angabe des `Backend` oder `Target` bereitzustellen. Um besser zu verstehen, wie der Transpiler funktioniert, erstelle einen Circuit und transpiliere ihn mit verschiedenen Informationen, wie folgt.

Importiere die notwendigen Bibliotheken und instanziiere den QPU:
Um einen abstrakten Circuit in einen ISA-Circuit umzuwandeln, der auf einem bestimmten Prozessor laufen kann, benötigt der Transpiler bestimmte Informationen über den Prozessor. Typischerweise sind diese Informationen im [`Backend`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.Backend#backend) oder [`Target`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.Target#target) gespeichert, der/das dem Transpiler übergeben wird, und es werden keine weiteren Informationen benötigt. Du kannst dem Transpiler aber auch explizit Informationen zur Verfügung stellen, zum Beispiel wenn du einen speziellen Anwendungsfall hast oder wenn du glaubst, dass diese Informationen dem Transpiler helfen, einen besser optimierten Circuit zu erzeugen.

Dieses Thema zeigt mehrere Beispiele für die Übergabe von Informationen an den Transpiler. Diese Beispiele verwenden das Target des Mock-Backends [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke).

In [1]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()
target = backend.target

Der Beispiel-Circuit verwendet eine Instanz von [`efficient_su2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2) aus Qiskits Circuit-Bibliothek.

In [2]:
from qiskit.circuit.library import efficient_su2

qc = efficient_su2(12, entanglement="circular", reps=1)

qc.draw("mpl")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/97f9acc1-ac53-4025-b413-485777932a9b-0.svg" alt="Output of the previous code cell" />

![Ausgabe der vorherigen Code-Zelle](../docs/images/guides/represent-quantum-computers/extracted-outputs/97f9acc1-ac53-4025-b413-485777932a9b-0.svg)

Dieses Beispiel verwendet Standardeinstellungen zum Transpilieren auf das `target` des `backend`, das alle Informationen liefert, die benötigt werden, um den Circuit in einen umzuwandeln, der auf dem Backend läuft.

In [3]:
from qiskit.transpiler import generate_preset_pass_manager

pass_manager = generate_preset_pass_manager(
    optimization_level=1, target=target, seed_transpiler=12345
)
qc_t_target = pass_manager.run(qc)
qc_t_target.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/4b81fb9d-d199-45c5-b119-c1f0b973afe9-0.svg" alt="Output of the previous code cell" />

![Ausgabe der vorherigen Code-Zelle](../docs/images/guides/represent-quantum-computers/extracted-outputs/4b81fb9d-d199-45c5-b119-c1f0b973afe9-0.svg)

Dieses Beispiel wird in späteren Abschnitten dieses Themas verwendet, um zu verdeutlichen, dass die Coupling Map und die Basis-Gates die wesentlichen Informationsstücke sind, die dem Transpiler für eine optimale Circuit-Konstruktion übergeben werden müssen. Der QPU kann in der Regel Standardeinstellungen für andere nicht übergebene Informationen wählen, wie z. B. Timing und Scheduling.

## Coupling Map

Die Coupling Map ist ein Graph, der anzeigt, welche Qubits miteinander verbunden sind und somit Zwei-Qubit-Gates zwischen ihnen haben. Manchmal ist dieser Graph gerichtet, was bedeutet, dass die Zwei-Qubit-Gates nur in eine Richtung verlaufen können. Der Transpiler kann jedoch jederzeit die Richtung eines Gates umkehren, indem er zusätzliche Einzel-Qubit-Gates hinzufügt. Ein abstrakter Quantum-Circuit kann immer auf diesem Graphen dargestellt werden, selbst wenn seine Konnektivität begrenzt ist, indem SWAP-Gates eingeführt werden, um die Quanteninformation zu verschieben.

Die Qubits unserer abstrakten Circuits werden _virtuelle Qubits_ genannt und die auf der Coupling Map _physikalische Qubits_. Der Transpiler stellt ein Mapping zwischen virtuellen und physikalischen Qubits bereit. Einer der ersten Schritte bei der Transpilation, die _Layout_-Phase, führt dieses Mapping durch.

> **Note:** Obwohl die Routing-Phase mit der _Layout_-Phase verknüpft ist — die die tatsächlichen Qubits auswählt — behandelt dieses Thema sie der Einfachheit halber standardmäßig als separate Phasen. Die Kombination aus Routing und Layout wird _Qubit-Mapping_ genannt. Weitere Informationen zu diesen Phasen findest du im Thema [Transpiler-Phasen](transpiler-stages).

Übergib das Schlüsselwortargument `coupling_map`, um seine Auswirkung auf den Transpiler zu sehen:

In [4]:
coupling_map = target.build_coupling_map()

pass_manager = generate_preset_pass_manager(
    optimization_level=0, coupling_map=coupling_map, seed_transpiler=12345
)
qc_t_cm_lv0 = pass_manager.run(qc)
qc_t_cm_lv0.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/ec354bee-e06b-42ea-a117-6c1a9308ca73-0.svg" alt="Output of the previous code cell" />

![Ausgabe der vorherigen Code-Zelle](../docs/images/guides/represent-quantum-computers/extracted-outputs/ec354bee-e06b-42ea-a117-6c1a9308ca73-0.svg)

Wie oben gezeigt, wurden mehrere SWAP-Gates eingefügt (jeweils bestehend aus drei CX-Gates), was auf aktuellen Geräten viele Fehler verursachen wird. Um zu sehen, welche Qubits auf der tatsächlichen Qubit-Topologie ausgewählt werden, verwende `plot_circuit_layout` aus den Qiskit-Visualisierungen:

In [5]:
from qiskit.visualization import plot_circuit_layout

plot_circuit_layout(qc_t_cm_lv0, backend, view="physical")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/9be74535-ed36-4d51-afeb-ee53c3f8a046-0.svg" alt="Output of the previous code cell" />

![Ausgabe der vorherigen Code-Zelle](../docs/images/guides/represent-quantum-computers/extracted-outputs/9be74535-ed36-4d51-afeb-ee53c3f8a046-0.svg)

Dies zeigt, dass unsere virtuellen Qubits 0–11 trivialerweise auf die Linie der physikalischen Qubits 0–11 abgebildet wurden. Kehren wir zur Standardeinstellung (`optimization_level=1`) zurück, die `VF2Layout` verwendet, wenn Routing erforderlich ist.

In [6]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, coupling_map=coupling_map, seed_transpiler=12345
)
qc_t_cm_lv1 = pass_manager.run(qc)
qc_t_cm_lv1.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/8035fd05-f7cd-4151-b19a-4968202246e6-0.svg" alt="Output of the previous code cell" />

![Ausgabe der vorherigen Code-Zelle](../docs/images/guides/represent-quantum-computers/extracted-outputs/8035fd05-f7cd-4151-b19a-4968202246e6-0.svg)

Jetzt werden keine SWAP-Gates mehr eingefügt und die ausgewählten physikalischen Qubits sind dieselben wie bei Verwendung der `target`-Klasse.

In [7]:
from qiskit.visualization import plot_circuit_layout

plot_circuit_layout(qc_t_cm_lv1, backend, view="physical")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/25d9fac3-abda-4b2d-81b4-351dc0772722-0.svg" alt="Output of the previous code cell" />

![Ausgabe der vorherigen Code-Zelle](../docs/images/guides/represent-quantum-computers/extracted-outputs/25d9fac3-abda-4b2d-81b4-351dc0772722-0.svg)

Jetzt liegt das Layout in einem Ring. Da dieses Layout die Konnektivität des Circuits respektiert, gibt es keine SWAP-Gates, was einen wesentlich besseren Circuit für die Ausführung liefert.

## Basis-Gates

Jeder Quantencomputer unterstützt einen begrenzten Befehlssatz, der als _Basis-Gates_ bezeichnet wird. Jedes Gate im Circuit muss in die Elemente dieses Satzes übersetzt werden. Dieser Satz sollte aus Einzel- und Zwei-Qubit-Gates bestehen, die einen universellen Gate-Satz bilden, was bedeutet, dass jede Quantenoperation in diese Gates zerlegt werden kann. Dies geschieht durch den [BasisTranslator](../api/qiskit/qiskit.transpiler.passes.BasisTranslator), und die Basis-Gates können dem Transpiler als Schlüsselwortargument übergeben werden, um diese Informationen bereitzustellen.

In [8]:
basis_gates = list(target.operation_names)
print(basis_gates)

['sx', 'switch_case', 'x', 'if_else', 'measure', 'for_loop', 'delay', 'ecr', 'id', 'reset', 'rz']


The default single-qubit gates on _ibm_sherbrooke_ are `rz`, `x`, and `sx`, and the default two-qubit gate is `ecr` (echoed cross-resonance). CX gates are constructed from `ecr` gates, so on some QPUs `ecr` is specified as the two-qubit basis gate, while on others `cx` is the default. The `ecr` gate is the _entangling_ part of the `cx` gate. In addition to the control gates, there are also `delay` and `measurement` instructions.


<Admonition type="note">
    QPUs have default basis gates, but you can choose whatever gates you want, as long as you provide the instruction or add pulse gates (see [Create transpiler passes.](custom-transpiler-pass)) The default basis gates are those that calibrations have been done for on the QPU, so no further instruction/pulse gates need to be provided. For example, on some QPUs `cx` is the default two-qubit gate and `ecr` on others. See the list of possible [native gates and operations](/docs/guides/qpu-information#native-gates) for more details.
</Admonition>

In [9]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1,
    coupling_map=coupling_map,
    basis_gates=basis_gates,
    seed_transpiler=12345,
)
qc_t_cm_bg = pass_manager.run(qc)
qc_t_cm_bg.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/313e4743-0.svg" alt="Output of the previous code cell" />

Die Standard-Einzel-Qubit-Gates auf _ibm_sherbrooke_ sind `rz`, `x` und `sx`, und das Standard-Zwei-Qubit-Gate ist `ecr` (echoed cross-resonance). CX-Gates werden aus `ecr`-Gates konstruiert, daher ist auf manchen QPUs `ecr` als Zwei-Qubit-Basis-Gate angegeben, während auf anderen `cx` der Standard ist. Das `ecr`-Gate ist der _verschränkende_ Teil des `cx`-Gates. Neben den Steuer-Gates gibt es auch `delay`- und `measurement`-Befehle.

> **Note:** QPUs haben Standard-Basis-Gates, aber du kannst beliebige Gates wählen, solange du die Anweisung bereitstellst oder Puls-Gates hinzufügst (siehe [Transpiler-Passes erstellen.](custom-transpiler-pass)). Die Standard-Basis-Gates sind diejenigen, für die Kalibrierungen auf dem QPU durchgeführt wurden, sodass keine weiteren Anweisungs-/Puls-Gates bereitgestellt werden müssen. Zum Beispiel ist auf manchen QPUs `cx` das Standard-Zwei-Qubit-Gate und auf anderen `ecr`. Weitere Details findest du in der Liste der möglichen [nativen Gates und Operationen](/guides/qpu-information#native-gates).

In [10]:
target["ecr"][(1, 0)]

InstructionProperties(duration=5.333333333333332e-07, error=0.007494257741828603)

![Ausgabe der vorherigen Code-Zelle](../docs/images/guides/represent-quantum-computers/extracted-outputs/313e4743-0.svg)

Beachte, dass die `CXGate`-Objekte in `ecr`-Gates und Einzel-Qubit-Basis-Gates zerlegt wurden.
## Geräte-Fehlerraten
Die `Target`-Klasse kann Informationen über die Fehlerraten für Operationen auf dem Gerät enthalten.
Zum Beispiel ruft der folgende Code die Eigenschaften für das echoed cross-resonance (ECR) Gate zwischen Qubit 1 und 0 ab (beachte, dass das ECR-Gate gerichtet ist):

In [11]:
from qiskit.transpiler import Target
from qiskit.circuit.controlflow import IfElseOp, SwitchCaseOp, ForLoopOp

err_targ = Target.from_configuration(
    basis_gates=basis_gates,
    coupling_map=coupling_map,
    num_qubits=target.num_qubits,
    custom_name_mapping={
        "if_else": IfElseOp,
        "switch_case": SwitchCaseOp,
        "for_loop": ForLoopOp,
    },
)

for i, (op, qargs) in enumerate(target.instructions):
    if op.name in basis_gates:
        err_targ[op.name][qargs] = target.instruction_properties(i)

Transpile with our new target `err_targ` as the target:

In [12]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, target=err_targ, seed_transpiler=12345
)
qc_t_cm_bg_et = pass_manager.run(qc)
qc_t_cm_bg_et.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/f1e270c4-e2cc-487e-a050-4180bc321b0b-0.svg" alt="Output of the previous code cell" />

Die Ausgabe zeigt die Dauer des Gates (in Sekunden) und seine Fehlerrate. Um dem Transpiler Fehlerinformationen bereitzustellen, erstelle ein Target-Modell mit den `basis_gates` und der `coupling_map` von oben und befülle es mit Fehlerwerten aus dem Backend `FakeSherbrooke`.